# 6.3 · 严谨回测：回测陷阱与可信度

> 回测最大的风险不是"没赚钱"，而是**回测赚钱、实盘亏钱**——绝大多数情况是踩到了回测陷阱。
> 本节系统梳理量化回测的经典偏差，以及如何做一份**可信**的回测。这些是面试和实战中最值钱的"避坑"经验。

## 一、未来函数（Look-ahead bias）

**做决策时用到了当时还不知道的信息。** 常见来源：
- 用当天的信号吃当天的收益（信号收盘后才产生，应次日成交）；
- 用了未来才发布的数据（财报、宏观数据）；
- 用全样本统计量（如用未来均值做标准化、用全样本选参数）；
- 复权处理不当（除权信息未来才发生）。

**避免：** 严格按时间推进；信号 shift；数据用"可得时间"而非"所属时间"对齐。

In [1]:
import numpy as np, pandas as pd
# 演示：未来函数 = 当天就知道当天涨跌（实际收盘后才知道）
np.random.seed(0)
r = pd.Series(np.random.randn(1000)/100)
# 未来函数：当天涨就持有、当天跌就空仓（等于盘前就知道收盘价，不可能）
wrong = (r > 0).astype(float) * r
# 正确：决策时不知道当天涨跌，只能每天持有（或用昨天及之前的信息）
right = r
def ann(x): return (1+x).prod()**(242/len(x))-1
print('未来函数（虚假，只吃正收益）年化收益: %.1f%%' % (ann(wrong)*100))
print('正确（无未来信息，每天持有）年化收益: %.1f%%' % (ann(right)*100))

未来函数（虚假，只吃正收益）年化收益: 143.8%
正确（无未来信息，每天持有）年化收益: -11.4%


## 二、幸存者偏差（Survivorship bias）

**股票池里只保留了"活到今天"的公司，剔除了退市、被合并的公司。**
- 退市股往往是因为业绩差、股价暴跌才退市；
- 只看幸存者，会高估收益（尤其小市值策略——很多小市值股最终退市归零）。

**避免：** 用历史时点真实的成分股（point-in-time universe），并纳入退市股的完整历史。
> 我们当前的沪深300 数据用的是"当前成分"，严格来说存在幸存者偏差；正式研究需用历史成分和退市数据。

## 三、数据对齐：财报的「报告期」vs「公告期」（PIT）

In [2]:
# 演示：财报在"报告期"结束后才公告，不能在报告期就用
fund = pd.DataFrame({
    '报告期': pd.to_datetime(['2020-03-31', '2020-06-30', '2020-09-30']),
    '公告日': pd.to_datetime(['2020-04-28', '2020-08-25', '2020-10-29']),
    'EPS': [0.30, 0.35, 0.40],
})
# 错误：按报告期对齐（3月底就用了一季报，但4月底才公告）
wrong = fund.set_index('报告期')['EPS']
# 正确：按公告日对齐（信息在公告日才可得）
right = fund.set_index('公告日')['EPS']
print('错误（报告期对齐，提前使用）:'); print(wrong)
print('\n正确（公告日对齐，PIT）:'); print(right)

错误（报告期对齐，提前使用）:
报告期
2020-03-31    0.30
2020-06-30    0.35
2020-09-30    0.40
Name: EPS, dtype: float64

正确（公告日对齐，PIT）:
公告日
2020-04-28    0.30
2020-08-25    0.35
2020-10-29    0.40
Name: EPS, dtype: float64


## 四、其他常见偏差

- **数据窥探 / 多重检验（data snooping）**：在同一批数据上反复试很多因子/参数，总能找到"历史有效"的，这是过拟合不是真规律。要用样本外、留出法、控制多重检验。
- **过拟合参数**：参数调到恰好拟合历史（如精确的窗口长度、阈值），样本外失效。参数要少、要有经济逻辑、要稳健性检验。
- **样本选择偏差**：只回测一段大牛市或特定市场状态，结论不具普适性。要覆盖牛熊、不同年份、不同市场。
- **忽略交易制度**：停牌（无法交易）、涨跌停（无法成交）、T+1（当日买入不能卖）、流动性（无对手盘）。
- **过度杠杆/资金假设**：假设能无限借钱、做空、忽略保证金和强平。

## 五、可信回测检查清单

1. 无未来函数：信号、数据、参数都只用当时可得信息；
2. 无幸存者偏差：历史成分 + 退市股；
3. 数据 PIT 对齐：财报用公告日，宏观用发布日；
4. 处理停牌、涨跌停、T+1、流动性；
5. 计入完整成本（佣金/印花税/滑点/冲击）；
6. 样本外验证 + 滚动/前推，避免数据窥探；
7. 覆盖足够长、跨牛熊的样本；
8. 参数稳健性（轻微改变参数结论不变）；
9. 与基准对比，看超额是否稳定、可解释；
10. 结果可复现（固定随机种子、代码化）。

> 一条经验：**回测越完美、越漂亮，越要警惕。** 真实策略都有缺陷、回撤和不完美。

## 六、小结与面试要点

- 回测陷阱：未来函数、幸存者偏差、数据窥探、过拟合、样本选择、忽略交易制度。
- 财报 PIT 对齐（公告日）、历史成分、退市股、成本、样本外是关键。
- 用检查清单保证回测可信；警惕"过于完美"的结果。

**面试高频：**
- 回测常见哪些偏差？（答：未来函数、幸存者偏差、数据窥探、过拟合）
- 什么是幸存者偏差？怎么处理？（答：只留存活股票高估收益；用历史成分和退市股）
- 财报数据怎么对齐？（答：按公告日而非报告期，PIT）
- 如何判断一个回测是否可信？（答：样本外、成本、跨牛熊、稳健性、无未来信息）

## 动手练习
1. 检查你之前的回测是否用了当前成分（幸存者偏差）；
2. 故意构造一个未来函数因子，看虚高程度；
3. 列出你当前研究流程中可能存在的 3 个偏差并想办法修正。

---
下一节 **6.4 绩效指标进阶：年化、夏普、Calmar、信息比率、盈亏比**。